# 03 - Model Training: Logistic Regression, Decision Tree & Random Forest

This notebook trains the three candidate credit scoring classifiers using the shared preprocessing pipeline and selects the top model based on weighted F1-Score.


In [ ]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.data_loader import DEFAULT_LABEL_MAPPING
from src.evaluate import compute_classification_metrics
from src.models import get_candidate_models
from src.preprocessing import load_preprocessed_data

data = load_preprocessed_data(os.path.abspath("../models/credit_features.joblib"))
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
class_labels = [DEFAULT_LABEL_MAPPING[i] for i in sorted(DEFAULT_LABEL_MAPPING.keys())]


In [ ]:
candidates = get_candidate_models(random_state=42, n_estimators=120)
summary_rows = []

for name, clf in candidates.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)
    metrics = compute_classification_metrics(y_test, y_pred, y_proba, class_labels)
    summary_rows.append({
        "Model": name,
        "Accuracy": round(metrics["accuracy"], 4),
        "Precision": round(metrics["weighted_precision"], 4),
        "Recall": round(metrics["weighted_recall"], 4),
        "F1-Score": round(metrics["weighted_f1_score"], 4),
        "ROC-AUC (OvR)": round(metrics["roc_auc_ovr_weighted"], 4),
    })

pd.DataFrame(summary_rows).sort_values(by="F1-Score", ascending=False)
